# Notebook 03 — Multi-Fidelity Successive Halving Search

**Session:** Hyper-parameter optimization  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Use `HalvingRandomSearchCV` to allocate small sample budgets (`n_samples`) to many
candidate configurations in early rounds, progressively eliminating weak candidates and
promoting only the strongest configurations to larger resource budgets.

In [ ]:
import time

from scipy.stats import loguniform
from sklearn import svm
from sklearn.datasets import make_classification
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingRandomSearchCV

## 1. Generate Synthetic Classification Dataset

In [ ]:
X, y = make_classification(
    n_samples=300,
    n_features=50,
    n_informative=10,
    n_redundant=25,
    n_repeated=15,
    n_clusters_per_class=5,
    flip_y=0.05,
    class_sep=0.5,
    random_state=0,
)

## 2. Successive Halving Over Sample Budgets

In [ ]:
svc = svm.SVC(probability=True, random_state=1)

search_distributions = {
    "kernel": ["linear", "rbf"],
    "C": loguniform(1, 1000),
    "gamma": loguniform(0.0001, 0.1),
}

search_func = HalvingRandomSearchCV(
    estimator=svc,
    param_distributions=search_distributions,
    resource="n_samples",
    max_resources=100,
    aggressive_elimination=True,
    scoring="accuracy",
    n_jobs=-1,
    cv=5,
    random_state=0,
)

start = time.perf_counter()
search_func.fit(X, y)
elapsed = time.perf_counter() - start

print(f"HalvingRandomSearchCV finished in {elapsed:.2f}s")
print("Best params:", search_func.best_params_)
print(f"Best 5-fold CV accuracy: {search_func.best_score_:.4f}")